# PSA_TreeWriter: PSA のヒットの TTree

PSA のモジュールが求めた量を、**1 ヒット(1 ボードの 1 ch)1 行**の TTree `PSA_TreeWriter/psa` に書きます。
`psa_decode.py` の `.h5` と `pixel_id.py` の `_pixel.h5` の中身を合わせたものに当たります。
`PSA_CenterGravity`、`PSA_Calibration` がチェーンに無いときは、その列は書きません。

| 列 | 型 | 中身 | 単位 | 作るモジュール |
|---|---|---|---|---|
| `sevt_id` | uint | `PSA_ReadData` のイベントの番号(0 から)。同じ値の行が同じトリガー | | PSA_ReadData |
| `trig` | uint | トリガーカウンタ(全ボード共通。`ReadData` と一緒のときは TPC のイベント ID と同じ) | | PSA_ReadData |
| `epoch` | int | トリガーカウンタが戻った回数(PSA だけのとき) | | PSA_ReadData |
| `clk` | uint | ボードのクロック | 1 µs | PSA_ReadData |
| `time` | double | ボードの最初のフレームからの時刻 | s | PSA_ReadData |
| `board` | short | HA のボード番号(`ha<NN>` の NN) | | PSA_ReadData |
| `ch` | short | ch 0〜5(= 5 − ADC の番号) | | PSA_ReadData |
| `hitnum` | short | このボードのこのイベントでヒットのある ch の数 | | PSA_ReadData |
| `hg`, `lg` | double | HG、LG の 4 本の積分の和 | ADC × サンプル | PSA_CalcFADC |
| `wave_sum[8]` | float | 8 本(HG0〜3、LG0〜3)それぞれの積分 | ADC × サンプル | PSA_CalcFADC |
| `ped[8]` | float | 8 本それぞれのペデスタル | ADC | PSA_CalcFADC |
| `peak_id` | short | ピークのサンプル | サンプル | PSA_CalcFADC |
| `satu_h`, `satu_l` | bool | HG、LG の飽和 | | PSA_CalcFADC |
| `temp[6]`, `temp_mean` | float | 温度センサー 6 個とその平均 | °C | PSA_CalcFADC |
| `x_hg`, `y_hg`, `x_lg`, `y_lg` | double | 4 隅の重心 | | PSA_CenterGravity |
| `x_corr`, `y_corr` | double | 補正後の位置(ピクセル n, m の中心が整数)。位置の較正の無い ch は NaN | ピクセル | PSA_Calibration |
| `pix_n`, `pix_m` | short | ピクセル 1〜8。8 × 8 の外は −1 | | PSA_Calibration |
| `pixel` | short | 64 ch + 8 (n − 1) + (m − 1)。外は −1 | | PSA_Calibration |
| `hg_corr` | double | 温度で直した hg = hg / gain(T)(温度ゲインの表が無いボードは hg と同じ) | ADC × サンプル | PSA_Calibration |
| `energy` | double | エネルギー = p0 + p1 × hg_corr。較正の無いときは −1 | keV | PSA_Calibration |
| `adc[400]` | ushort | 生の波形 `adc[サンプル × 8 + 本]`(`save_waveform: true` のときだけ) | ADC | PSA_ReadData |

In [ ]:
# ===== 入力と出力(絶対パス。データ量が大きいので、リポジトリの外の NAS などに置く)=====
from pathlib import Path
# 入力: UVSOR2026_late(killua/<ラン>, calltt/<ラン> があるディレクトリ)
DATA = Path("/mnt/d/UVSOR2026_late")
# 出力: チェーンの出力の置き場(チュートリアル間で共有)
CACHE = Path("/mnt/nas_eureka/onoda/<書き換え>/EricaNext_tutorial")

for p in (DATA, CACHE):
    if "<" in str(p):
        raise ValueError(f"パスを書き換えてください: {p}")
CACHE.mkdir(parents=True, exist_ok=True)
print("入力:", DATA)
print("出力:", CACHE)


In [1]:
import os, sys, json, subprocess
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import ROOT
ROOT.gROOT.SetBatch(True)

# リポジトリの場所、チェーンの実行スクリプト、設定ファイル(config/*.json)の読み込み
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "run_script" / "run_chain.py").exists())
RUN_CHAIN = REPO / "run_script" / "run_chain.py"
sys.path.insert(0, str(REPO / "config"))
import ericanext_config as ec

def th2(h):
    """ROOT の TH2 -> (中身 [x のビン, y のビン], x のビンの端, y のビンの端)"""
    nx, ny = h.GetNbinsX(), h.GetNbinsY()
    dt = np.float32 if h.InheritsFrom("TArrayF") else np.float64   # TH2F / TH2D
    a = np.frombuffer(h.GetArray(), dtype=dt, count=(nx + 2) * (ny + 2)).reshape(ny + 2, nx + 2).astype(float)
    ex = np.array([h.GetXaxis().GetBinLowEdge(i) for i in range(1, nx + 2)])
    ey = np.array([h.GetYaxis().GetBinLowEdge(i) for i in range(1, ny + 2)])
    return a[1:-1, 1:-1].T.copy(), ex, ey

def show_params(config, module):
    """設定ファイルの、あるモジュールのパラメータを、説明(_doc)と単位(_units)つきで表示する"""
    p = json.load(open(REPO / "config" / f"{config}.json"))["parameters"][module]
    doc, units = p.get("_doc", {}), p.get("_units", {})
    for k, v in p.items():
        if k.startswith("_"):
            continue
        u = f" [{units[k]}]" if k in units else ""
        print(f"{k} = {json.dumps(v, ensure_ascii=False)}{u}\n    {doc.get(k, '')}")

## 1. データの準備

2026-11-03 の UVSOR での **HA_cal のラン `20261103_run01`**(PSA だけ、14 ボード、約 840 MB)を使います。
位置の較正(`calib/psa/position_haNN.csv`)を作ったときと同じ日の較正データです。
データは解析用の計算機の D ドライブ(`/mnt/d/UVSOR2026_late`)にあります。
ほかの場所にあるときは、冒頭の `DATA` を `UVSOR2026_late` に当たるディレクトリの絶対パスに書き換えてください。

1 つのランのボードは、2 つの storage に分かれて書かれています。

| storage | ディレクトリ | PSA のボード |
|---|---|---|
| storage_g1 | `killua/<ラン>` | ha00〜ha05 |
| storage_g0 | `calltt/<ラン>` | ha06〜ha16 |

`run_chain.py` には、片方を `--data-dir`、もう片方を `PSA_ReadData.extra_data_dirs` で渡します。

In [2]:
RUN = "20261103_run01"
RUN_DIRS = [DATA / "killua" / RUN, DATA / "calltt" / RUN]
for d in RUN_DIRS:
    print("OK     " if d.is_dir() else "MISSING", d)

OK      /mnt/d/UVSOR2026_late/killua/20261103_run01
OK      /mnt/d/UVSOR2026_late/calltt/20261103_run01


## 2. チェーンを流す

設定ファイル `smile3_psa`(`PSA_ReadData → PSA_CalcFADC → PSA_CenterGravity → PSA_Calibration → PSA_TreeWriter`)で、ラン全体を流します(40 秒ほど)。
ターミナルでは、リポジトリの `run_script` で次を実行するのと同じです。

```bash
python3 run_chain.py smile3_psa -o psa_20261103_run01.root \
    --data-dir /mnt/d/UVSOR2026_late/killua/20261103_run01 \
    --set 'PSA_ReadData.extra_data_dirs=["/mnt/d/UVSOR2026_late/calltt/20261103_run01"]'
```

出力は `<CACHE>/psa_cal_<ラン>.root`(`CACHE` は冒頭で指定)に置き、既にあれば流し直しません(PSA のチュートリアルと `4_MakeCalibration/PSA_Energy` で共有)。
`calib/psa` の較正ファイルを変えたときは、このファイルを消して流し直してください。

In [3]:
def run_chain(out, *args):
    """run_chain.py を実行する(出力が既にあれば何もしない)"""
    cmd = [sys.executable, str(RUN_CHAIN), "smile3_psa", "-o", str(out), "--data-dir", str(RUN_DIRS[0]),
           "--set", "PSA_ReadData.extra_data_dirs=" + json.dumps([str(RUN_DIRS[1])]), *args]
    print(" ".join(cmd))
    if not out.exists():
        r = subprocess.run(cmd, capture_output=True, text=True)
        if r.returncode != 0:
            print(r.stdout[-3000:], r.stderr[-3000:])
        r.check_returncode()

OUT = CACHE / f"psa_cal_{RUN}.root"   # 今の calib/psa(位置・温度ゲイン・エネルギー)で流したもの
run_chain(OUT)

/usr/bin/python3 /home/onoda/ANLNext/Erica_git_dev/EricaNext/run_script/run_chain.py smile3_psa -o /home/onoda/ANLNext/Erica_git_dev/EricaNext/tutorials/cache/psa_cal_20261103_run01.root --data-dir /mnt/d/UVSOR2026_late/killua/20261103_run01 --set PSA_ReadData.extra_data_dirs=["/mnt/d/UVSOR2026_late/calltt/20261103_run01"]


## 3. ツリーの中身

ROOT ファイルに入っている列と型を表示します。

In [4]:
f = ROOT.TFile(str(OUT))
tree = f.Get("PSA_TreeWriter/psa")
print("entries:", tree.GetEntries())
for br in tree.GetListOfBranches():
    print(f"  {br.GetName():10s} {br.GetTitle()}")

entries: 862319


  sevt_id    sevt_id/i
  trig       trig/i
  epoch      epoch/I
  clk        clk/i
  time       time/D
  board      board/S
  ch         ch/S
  hitnum     hitnum/S
  hg         hg/D
  lg         lg/D
  wave_sum   wave_sum[8]/F
  ped        ped[8]/F
  peak_id    peak_id/S
  satu_h     satu_h/O
  satu_l     satu_l/O
  temp       temp[6]/F
  temp_mean  temp_mean/F
  x_hg       x_hg/D
  y_hg       y_hg/D
  x_lg       x_lg/D
  y_lg       y_lg/D
  x_corr     x_corr/D
  y_corr     y_corr/D
  pix_n      pix_n/S
  pix_m      pix_m/S
  pixel      pixel/S
  hg_corr    hg_corr/D
  energy     energy/D


## 4. 読み方

`RDataFrame` で読みます。`Filter` で選んでから `AsNumpy` で numpy の配列にするのが簡単です。
`short` の列は numpy の object の配列になるので `astype(int)` で直し、配列の列(`wave_sum` など)は `np.stack` で 2 次元にします。

In [5]:
df = ROOT.RDataFrame("PSA_TreeWriter/psa", str(OUT))
d = df.Filter("board == 3 && pixel >= 0 && !satu_h").AsNumpy(["sevt_id", "ch", "pix_n", "pix_m", "hg", "wave_sum"])
ch = d["ch"].astype(int)
wave_sum = np.stack([np.asarray(v) for v in d["wave_sum"]])
print(len(ch), "hits of ha03 in 8x8 without saturation; wave_sum:", wave_sum.shape)
print("hits per ch:", np.bincount(ch))

65622 hits of ha03 in 8x8 without saturation; wave_sum: (65622, 8)
hits per ch: [11635 11539 10626 10877  9711 11234]


1 イベントのヒットをまとめるときは `sevt_id` で並べます。下は、2 つ以上のボードにヒットがあったイベントの例です。

In [6]:
a = df.AsNumpy(["sevt_id", "trig", "board", "ch", "hg", "pixel"])
sid = a["sevt_id"]
u, cnt = np.unique(sid, return_counts=True)
multi = u[cnt >= 2][:5]
for e in multi:
    m = sid == e
    print(f"sevt_id {e} (trig {a['trig'][m][0]}):", [(f"ha{int(b):02d}", int(c), round(float(h)), int(p)) for b, c, h, p in zip(a['board'][m], a['ch'][m], a['hg'][m], a['pixel'][m])])

sevt_id 0 (trig 1): [('ha02', 3, 36138, 208), ('ha04', 1, 25979, 65), ('ha07', 5, 39253, 366)]
sevt_id 1 (trig 2): [('ha04', 1, 34254, 66), ('ha09', 5, 31144, 368), ('ha13', 5, 13339, -1)]
sevt_id 2 (trig 3): [('ha01', 4, 28782, -1), ('ha02', 5, 5141, 358), ('ha04', 1, 28805, 75), ('ha06', 1, 26181, 87), ('ha09', 1, 10776, -1)]
sevt_id 3 (trig 4): [('ha04', 1, 49656, 74), ('ha06', 2, 44345, 170), ('ha15', 0, 55959, 18)]
sevt_id 5 (trig 6): [('ha02', 0, 52116, 61), ('ha03', 5, 53835, 353), ('ha04', 1, 4085, -1), ('ha09', 5, 28476, 339), ('ha13', 3, 12996, 243)]


## 5. 波形の列 `adc`

`PSA_TreeWriter.save_waveform: true` で、生の波形 `adc[400]` も書きます(1 ヒット 800 バイト増えます)。
読み方は [3_PSA_CalcFADC](../3_PSA_CalcFADC/PSA_CalcFADC.ipynb) の 3 節を見てください。

## 6. パラメータ

In [7]:
show_params("smile3_psa", "PSA_TreeWriter")

save_waveform = false
    true: 波形(50 サンプル × HG0-3, LG0-3 の生の ADC 値)も adc[400] に保存する
